# QuantLib
Zeke Romero

[QuantLib](https://github.com/lballabio/QuantLib) is a free, OS library that banks and investors use to work out what financial contracts are worth. It is written in C++, and in this notebook we use it from Python through its official bindings (QuantLib-SWIG).

The interesting part for this class bc almost every calculation QuantLib does is a numerical method, such as root-finding, interpolation, solving differential equations on a grid, and estimating answers with random sampling.


## 1. What problem does it solve x Who is involved?

**A quick finance explanation (needed for understanding):** an **option** is a contract that gives you the right to buy a stock at a set price (the **strike**) on a future date. If the stock ends up above the strike, the option pays off. If not, it is worth nothing. The hard question is: 

what is that right worth today (you have to pay upfront for the contract)?

The answer depends on how much the stock price tends to bounce around, which is called **volatility**.

**What QuantLib does:** You give it a contract and market data, and it returns a price. Behind the scenes that means:

- **Root-finding**: working backwards from a price to the volatility that produces it
- **Interpolation**: filling in interest rates between the handful of dates where we know them
- **Differential equations**: the price of an option satisfies a known equation that QuantLib solves on a grid
- **Random sampling (Monte Carlo)**: simulating thousands of possible futures for the stock and averaging the payoffs

To buy an option you pay a price upfront for the contract, this library calculates what that upfront price should be using methods above, if the price of your contract is lower then your in good standing, if its higher then you should sell or not buy.

Qauntlibs answer is only as good as the volatility you give it... (exampkle in SEC 3)

What makes it unique is that it is the standard free reference library for this whole field, and every method sits behind the same interface, so you can swap one method for another with one line of code.

**Who is involved**

- **Who builds it:** one lead maintainer, Luigi Ballabio, plus about 100 contributors in the past year (about 370 in total)
- **Who uses it:** analysts and risk managers at banks, hedge funds and financial software companies
- **Who pays for it:** no one funds it officially, contributors are mostly paid by their own employers or volunteer
- **Who is affected:** the customers and regulators who rely on the prices and risk numbers those firms produce
- **How they work together:** on GitHub (about 80 comments and 29 code changes per week) and through [mailing lists](https://www.quantlib.org/mailinglists.shtml). Every change is reviewed by a person and checked by automated tests before it is accepted.

### How it all fits together (most important seciton)


Using QuantLib to price an option comes down to 3 steps. Each one is shown later in this notebook


1: Find the volatility (method: root-finding)

- **Input:** the market prices of popular options that trade a lot
- **Output:** the implied volatility, meaning how jumpy the market thinks the stock will be
- **Why:** volatility can't be looked up like a stock price. Root-finding works backwards from a price to the volatility that produces it
- **In this notebook:** section 3, *Conditioning*

2: Check you can trust it (concept: conditioning)

- **Input:** the volatility(step 1), and the fact that prices are only known to the nearest cent
- **Output:** how much a tiny price error changes the volatility
- **Why:** this is a check, not a method. Volatilities from options with a strike near the stock price are reliable; ones far away are shaky, so traders lean on the reliable ones.
- **In this notebook:**  section 3, *Conditioning*

3: CAlc. the price (methods: exact formula, Monte Carlo, finite differences)

- **Input:** the contract (strike and end date) plus market data (stock price, interest rate, and the volatility from step 1)
- **Output:** the fair upfront price of the option, plus risk numbers such as delta (how much the price moves if the stock goes up $1)
- **Why:** so you can price contracts the market doesn't clearly price, and compare against the market price- if the market price is lower, it's a good deal -- if higher, sell or don't buy
- **In this notebook:** section 2, the tutorial, where one option is priced with many methods



**IMPORTanT** QuantLib's answer is only as good as the volatility you give it.


## 2. Running an example from the project

QuantLibs own european-option.py example from [QuantLib-SWIG](https://github.com/lballabio/QuantLib-SWIG/tree/master/Python/examples), 

It prices 1 option using many different methods, then shows how far each method's answer is from the exact answer. Read the table like this: Option value is each method's price, and Actual error is how far off it is (smaller is better).

# European options (QAUNTLIB EX)


In [1]:
import QuantLib as ql
import pandas as pd

### Global parameters

In [2]:
todaysDate = ql.Date(15, ql.May, 1998)
ql.Settings.instance().evaluationDate = todaysDate

In [3]:
interactive = 'get_ipython' in globals()

### Option construction

In [4]:
exercise = ql.EuropeanExercise(ql.Date(17, ql.May, 1999))
payoff = ql.PlainVanillaPayoff(ql.Option.Call, 8.0)

In [5]:
option = ql.VanillaOption(payoff, exercise)

### Market data

In [6]:
underlying = ql.SimpleQuote(7.0)
dividendYield = ql.FlatForward(todaysDate, 0.05, ql.Actual365Fixed())
volatility = ql.BlackConstantVol(todaysDate, ql.TARGET(), 0.10, ql.Actual365Fixed())
riskFreeRate = ql.FlatForward(todaysDate, 0.05, ql.Actual365Fixed())

### Processes and models

In [7]:
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(underlying),
    ql.YieldTermStructureHandle(dividendYield),
    ql.YieldTermStructureHandle(riskFreeRate),
    ql.BlackVolTermStructureHandle(volatility),
)

In [8]:
hestonProcess = ql.HestonProcess(
    ql.YieldTermStructureHandle(riskFreeRate),
    ql.YieldTermStructureHandle(dividendYield),
    ql.QuoteHandle(underlying),
    0.1 * 0.1,
    1.0,
    0.1 * 0.1,
    0.0001,
    0.0,
)
hestonModel = ql.HestonModel(hestonProcess)

### Pricing


In [9]:
results = []

#### Analytic formula

In [10]:
option.setPricingEngine(ql.AnalyticEuropeanEngine(process))
value = option.NPV()
refValue = value

results.append(('Analytic', value, None, None))

#### Heston semi-analytic formula

In [11]:
option.setPricingEngine(ql.AnalyticHestonEngine(hestonModel))
value = option.NPV()

results.append(('Heston analytic', value, None, abs(value - refValue)))

#### Heston COS method

In [12]:
option.setPricingEngine(ql.COSHestonEngine(hestonModel))
value = option.NPV()

results.append(('Heston COS', value, None, abs(value - refValue)))

#### Integral method

In [13]:
option.setPricingEngine(ql.IntegralEngine(process))
value = option.NPV()

results.append(('Integral', value, None, abs(value - refValue)))

#### Finite-difference method

In [14]:
timeSteps = 801
gridPoints = 800

In [15]:
option.setPricingEngine(ql.FdBlackScholesVanillaEngine(process, timeSteps, gridPoints))
value = option.NPV()

results.append(('Finite diff.', value, None, abs(value - refValue)))

#### Binomial method

In [16]:
timeSteps = 801

In [17]:
for tree in ["JR", "CRR", "EQP", "Trigeorgis", "Tian", "LR", "Joshi4"]:
    option.setPricingEngine(ql.BinomialVanillaEngine(process, tree, timeSteps))
    value = option.NPV()

    results.append(('Binomial (%s)' % tree, value, None, abs(value - refValue)))

#### Monte Carlo method

In [18]:
option.setPricingEngine(ql.MCEuropeanEngine(process, "pseudorandom", timeSteps=1,
                                            requiredTolerance=0.02, seed=42))
value = option.NPV()

results.append(("Monte Carlo (pseudo-random)", value, option.errorEstimate(), abs(value - refValue)))

In [19]:
option.setPricingEngine(ql.MCEuropeanEngine(process, "lowdiscrepancy", timeSteps=1,
                                            requiredSamples=32768))
value = option.NPV()

results.append(("Monte Carlo (low-discrepancy)", value, None, abs(value - refValue)))

### Results

In [20]:
df = pd.DataFrame(results,
                  columns=["Method", "Option value", "Error estimate", "Actual error"])

In [21]:
df.style.hide(axis="index")

Method,Option value,Error estimate,Actual error
Analytic,0.030334,nan,nan
Heston analytic,0.030334,nan,0.000000
Heston COS,0.030334,nan,0.000000
Integral,0.030334,nan,0.000000
Finite diff.,0.030334,nan,0.000000
Binomial (JR),0.030287,nan,0.000047
Binomial (CRR),0.030342,nan,0.000007
Binomial (EQP),0.030392,nan,0.000058
Binomial (Trigeorgis),0.030342,nan,0.000007
Binomial (Tian),0.030303,nan,0.000031


## 3. Interesting things to me

### Accuracy vs. cost

The tutorial's table prices the same option with many methods, so we can compare how accurate each one is (the Actual error column, smaller is better):

- **Exact and near-exact methods** (Heston formulas, the integral method, finite differences) are accurate to about 7 decimal places or better
- **Binomial trees** (a simple step-by-step model of the stock going up or down) are accurate to about 4 or 5 decimal places, and some versions far better
- **Monte Carlo** (random simulations) is the least accurate here. It gets more accurate with more simulations, but only by doing a lot more work

So why keep Monte Carlo at all? For contracts that depend on many stocks at once, grid methods become impossibly large, while Monte Carlo's cost barely changes. That is why QuantLib includes so many methods, each one a different trade-off between speed and accuracy, and you can swap between them with one line of code.

### Conditioning

Traders often quote options not by price but by implied volatility (the volatility that would make the formula produce the price they see). Getting it means working backwards from a price, which QuantLib does with a root finder.

Note: this tells you what volatility the market is already assuming, not the "right" one, to decide whether an option is a good deal, a trader compares it to their own estimate, usually based on how jumpy the stock has been in the past.

A problem is well-conditioned if a tiny error in the input causes only a tiny error in the answer. Prices are only known to the nearest cent, and that matters here:

- When the strike is close to the stock price, the option's price reacts strongly to volatility, so working backwards is reliable: a 1-cent error barely changes the answer (well-conditioned).
- When the strike is far from the stock price, the price hardly reacts to volatility at all, so working backwards amplifies a 1-cent error into a huge volatility error (ill-conditioned).

**To get the volatility for a far away strike, use the volatility from well-conditioned options (strikes near the stock price) instead of trusting the far away option’s own answer, traders will fit a smooth curve that’s anchored by the reliable near strikes and read the far strike’s volatility off that curve to get reliable answers, they create a line to get there answer, because the root finder could have noise**

This is a property of the question itself, not of QuantLib's root-finder. No better algo can fix it.




## 4. A question I have

QuantLib gives you many methods for pricing the same contract, and as the table above shows, each method trades speed for accuracy differently, so the best choice depends on the contract and how much speed or accuracy you need.

How do users decide which method to use?

Do they follow certain rules (ex. a grid for one stock, Monte Carlo for many), or run several methods and compare the answers? 

Can methods be combined? (ex. using a fast exact formula to cancel out most of Monte Carlo's random error?)

## 5. Which method is best proposla?

IDEA-The tutorial's table shows how accurate each pricing method is, but not how long each one takes. In practice we need both, a trader quoting prices live needs speed, while a risk team running overnight reports needs accuracy. We would measure both and put them on one plot.

What we would measure:

1. Price the same option with several methods: finite differences, binomial trees, Monte Carlo, and low-discrepancy Monte Carlo
2. Run each method at several sizes (more grid points, more tree steps, or more sims). Bigger sizes should be more accurate but slower
3. For every run, record two numbers:
   - **Time:** how long it took to get the price
   - **Error:** how far the price is from the exact answer 

**The figure**

- **x-axis:** time to compute one price (milliseconds, log scale)
- **y-axis:** error in the price (dollars, log scale)
- **One line per method**, with each point on the line being one size

**How to read it:** lower is more accurate and further left is faster, so the line closest to the bottom left corner is the best method. A dashed line at 1 cent shows which methods reach "good enough" accuracy, and how fast


What is learned: a clear answer to which method should I use, and when? 
